In [45]:
import requests
import zipfile
import io
from pathlib import Path

# The dataset is taken from Mendeley Data, which is a platform for sharing research data. The dataset is provided as a ZIP file, which contains a CSV file with the email spam detection data.
# All rights to the dataset belong to the original authors, and it is used here for educational and research purposes only.
# To access the dataset, visit: https://data.mendeley.com/datasets/f45bkkt8pr/1
url = (
    "https://data.mendeley.com/public-files/datasets/f45bkkt8pr/files/edb361de-918d-469f-9106-e84823830665/file_downloaded"
)

# Download ZIP
response = requests.get(url, timeout=60)
response.raise_for_status()

# Extract ZIP
with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    z.extractall("data")
    print(z.namelist())

['Dataset_5971.csv']


In [46]:
import pandas as pd

raw_df = pd.read_csv("data/Dataset_5971.csv")

"""
df.head() gives a preview of the first few rows of the DataFrame, 
allowing to quickly inspect the data and its structure.
"""
raw_df.head()

,LABEL,TEXT,URL,EMAIL,PHONE
0,ham,Your opinion about me? 1. Over 2. Jada 3. Kusr...,No,No,No
1,ham,What's up? Do you want me to come online? If y...,No,No,No
2,ham,So u workin overtime nigpun?,No,No,No
3,ham,"Also sir, i sent you an email about how to log...",No,No,No
4,Smishing,Please Stay At Home. To encourage the notion o...,No,No,No


In [47]:
# get feature column names
raw_df.columns

Index(['LABEL', 'TEXT', 'URL', 'EMAIL', 'PHONE'], dtype='object')

In [48]:
# Going to extract only Label and Message feature columns from dataset.
df = raw_df[["TEXT", "LABEL"]].copy()

"""
Steps of data preprocessing:
1. original: " Ham "
2. str.lower() = lowercased: " ham "
3. str.strip() = stripped: "ham"
4. map() = mapped to integer: 0
"""
df["LABEL"] = df["LABEL"].str.lower().str.strip().map({"ham": 0, "spam": 1, "smishing": 2})

# Get a random sample of one row from each group in the DataFrame.
df.groupby("LABEL").sample(n=1, random_state=42)



,TEXT,LABEL
4870,Finished class where are you.,0
2072,Get your garden ready for summer with a FREE s...,1
1662,"Dear Customer,U've been selected to stay in F1...",2


In [ ]:
# Always start from the original dataset
df = raw_df[["TEXT", "LABEL"]].copy()

# Encode labels
df["LABEL"] = df["LABEL"].str.lower().str.strip().map({
    "ham": 0,
    "spam": 1,
    "smishing": 2
})

assert df["LABEL"].notna().all()

print(f"Original dataset: {len(df)}")

# Remove missing values
before = len(df)
df = df.dropna(subset=["TEXT", "LABEL"]).copy()
print(f"Missing rows removed: {before - len(df)}")

# Strip whitespace
df["TEXT"] = df["TEXT"].str.strip()

# Remove empty messages
before = len(df)
df = df[df["TEXT"] != ""].copy()
print(f"Empty rows removed: {before - len(df)}")

# Normalise whitespace
df["TEXT"] = df["TEXT"].str.replace(r"\s+", " ", regex=True)

# Remove conflicting messages
before = len(df)

# Nunique counts of LABEL for each unique TEXT. 
# If a TEXT has more than one unique LABEL, it is considered conflicting.
conflicts = df.groupby("TEXT")["LABEL"].nunique()
conflicting_texts = conflicts[conflicts > 1].index

df = df[~df["TEXT"].isin(conflicting_texts)].copy()

print(f"Conflicting rows removed: {before - len(df)}")

# Remove duplicates
before = len(df)
df = df.drop_duplicates(subset=["TEXT"])

print(f"Duplicate rows removed: {before - len(df)}")

# Reset index
df = df.reset_index(drop=True)

print(f"\nFinal dataset: {len(df)}")
print(df["LABEL"].value_counts())

Original dataset: 5971
Missing rows removed: 0
Empty rows removed: 0
Conflicting rows removed: 68
Duplicate rows removed: 106

Final dataset: 5797
LABEL
0    4834
2     528
1     435
Name: count, dtype: int64


In [52]:
!pip install scikit-learn

In [53]:
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets using 70% of the data for training and 30% for testing. 
# The stratify parameter ensures that the class distribution is preserved in both sets.
# Example: If the original dataset has 70% ham and 30% spam, the training and testing sets will also have approximately the same distribution.
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df["LABEL"])

val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["LABEL"])

In [54]:
for name, split in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Testing", test_df)
]:
    print(f"\n{name}: {len(split)} samples")
    print(split["LABEL"].value_counts())


Training: 4057 samples
LABEL
0    3383
2     370
1     304
Name: count, dtype: int64

Validation: 870 samples
LABEL
0    725
2     79
1     66
Name: count, dtype: int64

Testing: 870 samples
LABEL
0    726
2     79
1     65
Name: count, dtype: int64


In [ ]:
# Export the dataset locally as CSV

data_dir = Path("data")
unprocessed_dir = data_dir / "unprocessed"
processed_dir = data_dir / "processed"

# Create directories if they don't exist
unprocessed_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

# Save the unprocessed dataset
raw_df.to_csv(unprocessed_dir / "Dataset_5971.csv", index=False)

# Save the processed datasets
train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir / "val.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

In [57]:
# Get access to Google Drive to save the datasets there as well
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [58]:
data_dir_drive = Path("/content/drive/MyDrive/SMS-Spam-Detection/data")
unprocessed_dir_drive = data_dir_drive / "unprocessed"
processed_dir_drive = data_dir_drive / "processed"

# Create directories if they don't exist
unprocessed_dir_drive.mkdir(parents=True, exist_ok=True)
processed_dir_drive.mkdir(parents=True, exist_ok=True)

# Save the unprocessed dataset to Google Drive
raw_df.to_csv(unprocessed_dir_drive / "Dataset_5971.csv", index=False)

# Save the processed datasets to Google Drive
train_df.to_csv(processed_dir_drive / "train.csv", index=False)
val_df.to_csv(processed_dir_drive / "val.csv", index=False)
test_df.to_csv(processed_dir_drive / "test.csv", index=False)